# Practica Prueba técnica Globant

## Sección 2
### SQL — JOINS | WINDOW FUNCTIONS

In [50]:
%pip install ipython-sql duckdb duckdb-engine sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [51]:
import duckdb 
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [52]:
np.random.seed(42)

In [53]:
companies = ['Company A', 'Company B', 'Company C', 'Company D', 'Company E']
num_records = 10000

df_purchases = pd.DataFrame({
    'customer_id': np.random.randint(1, 1001, num_records),
    'company': np.random.choice(companies, num_records),
    'is_premium': np.random.choice([True, False], num_records, p=[0.3, 0.7]),
    'purchase_date': [datetime(2023,1,1) + timedelta(days=random.randint(0, 365)) for _ in range(num_records)],
    'purchase_amount': np.round(np.random.uniform(10, 500, num_records), 2)
    })

print(df_purchases.head())
print(df_purchases.info())
print(df_purchases.shape)
print(df_purchases['company'].value_counts())
print(df_purchases['is_premium'].value_counts())



   customer_id    company  is_premium purchase_date  purchase_amount
0          103  Company B        True    2023-11-25           465.11
1          436  Company C       False    2023-08-08           175.05
2          861  Company A       False    2023-05-14           136.11
3          271  Company D       False    2023-08-18           365.78
4          107  Company B       False    2023-10-03           356.33
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      10000 non-null  int32         
 1   company          10000 non-null  object        
 2   is_premium       10000 non-null  bool          
 3   purchase_date    10000 non-null  datetime64[ns]
 4   purchase_amount  10000 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(1), int32(1), object(1)
memory usage: 283.3+ KB
None
(10000, 5)


In [54]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///my_data_seccion2.db")

df_purchases.to_sql("purchases", con=engine, if_exists="replace", index=False)

10000

In [55]:
%sql sqlite:///my_data_seccion2.db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [56]:
%%sql
SELECT *
FROM purchases
ORDER BY purchase_date ASC
LIMIT 5;

 * sqlite:///my_data_seccion2.db
Done.


customer_id,company,is_premium,purchase_date,purchase_amount
932,Company B,1,2023-01-01 00:00:00.000000,142.62
541,Company C,1,2023-01-01 00:00:00.000000,186.87
37,Company C,0,2023-01-01 00:00:00.000000,137.21
570,Company E,0,2023-01-01 00:00:00.000000,368.37
944,Company C,0,2023-01-01 00:00:00.000000,381.13


In [57]:
%%sql
SELECT customer_id, company, is_premium, purchase_date, purchase_amount,
SUM(purchase_amount) OVER (PARTITION BY company) AS total_spent_in_company,
ROUND(purchase_amount / SUM(purchase_amount) OVER (PARTITION BY company) * 100, 2) AS percentage_of_total,
SUM(purchase_amount) OVER (PARTITION BY customer_id ORDER BY purchase_date) AS total_spent_by_customer,
ROUND(AVG(purchase_amount) OVER (PARTITION BY customer_id), 2) AS average_purchase_by_customer
FROM purchases
ORDER BY purchase_date, customer_id, company
LIMIT 10;


 * sqlite:///my_data_seccion2.db
Done.


customer_id,company,is_premium,purchase_date,purchase_amount,total_spent_in_company,percentage_of_total,total_spent_by_customer,average_purchase_by_customer
37,Company C,0,2023-01-01 00:00:00.000000,137.21,485795.95,0.03,137.21,149.78
62,Company A,1,2023-01-01 00:00:00.000000,343.94,513412.62,0.07,343.94,264.55
73,Company A,0,2023-01-01 00:00:00.000000,194.32,513412.62,0.04,194.32,257.38
134,Company D,0,2023-01-01 00:00:00.000000,337.1,514233.35,0.07,337.1,271.68
168,Company B,0,2023-01-01 00:00:00.000000,34.18,536214.02,0.01,34.18,192.62
173,Company D,0,2023-01-01 00:00:00.000000,433.0,514233.35,0.08,433.0,242.68
184,Company A,0,2023-01-01 00:00:00.000000,116.84,513412.62,0.02,116.84,221.32
191,Company A,0,2023-01-01 00:00:00.000000,127.82,513412.62,0.02,127.82,256.95
199,Company A,0,2023-01-01 00:00:00.000000,258.56,513412.62,0.05,258.56,318.99
207,Company E,0,2023-01-01 00:00:00.000000,335.45,521298.33,0.06,335.45,291.02


In [58]:
%%sql
WITH filas AS (
    SELECT customer_id, company, is_premium, purchase_date, purchase_amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY purchase_date ASC) AS number_of_purchase,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY purchase_date DESC) AS last_purchase
FROM purchases
)

SELECT *
FROM filas
WHERE number_of_purchase = 1
ORDER BY purchase_date ASC
LIMIT 15;

 * sqlite:///my_data_seccion2.db
Done.


customer_id,company,is_premium,purchase_date,purchase_amount,number_of_purchase,last_purchase
37,Company C,0,2023-01-01 00:00:00.000000,137.21,1,6
62,Company A,1,2023-01-01 00:00:00.000000,343.94,1,6
73,Company A,0,2023-01-01 00:00:00.000000,194.32,1,19
134,Company D,0,2023-01-01 00:00:00.000000,337.1,1,11
168,Company B,0,2023-01-01 00:00:00.000000,34.18,1,11
173,Company D,0,2023-01-01 00:00:00.000000,433.0,1,11
184,Company A,0,2023-01-01 00:00:00.000000,116.84,1,10
191,Company A,0,2023-01-01 00:00:00.000000,127.82,1,13
199,Company A,0,2023-01-01 00:00:00.000000,258.56,1,8
207,Company E,0,2023-01-01 00:00:00.000000,335.45,1,13


In [59]:
%%sql
WITH ganancias_mensuales AS (
    SELECT strftime('%Y-%m', purchase_date) AS month, 
           company, 
           SUM(purchase_amount) AS total_revenue
    FROM purchases
    GROUP BY month, company
)

SELECT month, company, total_revenue,
RANK() OVER (PARTITION BY month ORDER BY total_revenue DESC) AS revenue_rank
FROM ganancias_mensuales
WHERE month >= '2023-01' AND month <= '2023-02'
ORDER BY month, revenue_rank
LIMIT 15;

 * sqlite:///my_data_seccion2.db
Done.


month,company,total_revenue,revenue_rank
2023-01,Company B,45262.94,1
2023-01,Company C,41799.95,2
2023-01,Company E,41789.77,3
2023-01,Company A,39931.04,4
2023-01,Company D,36497.75,5
2023-02,Company E,39827.75,1
2023-02,Company D,39613.99,2
2023-02,Company A,38957.45,3
2023-02,Company B,35254.85,4
2023-02,Company C,33209.76,5


In [73]:
%%sql
WITH ganancias_mensuales AS (
    SELECT strftime('%Y-%m', purchase_date) AS month, 
           company, 
           is_premium,
           SUM(purchase_amount) AS total_revenue,
           COUNT(DISTINCT customer_id) AS unique_customers
    FROM purchases
    GROUP BY month, company, is_premium
)

SELECT month, company, is_premium, total_revenue, unique_customers,
LAG(total_revenue) OVER (PARTITION BY company, is_premium ORDER BY month) AS previous_month_revenue,
ROUND(total_revenue - LAG(total_revenue) OVER (PARTITION BY company, is_premium ORDER BY month), 2) AS revenue_change,
ROUND((total_revenue - LAG(total_revenue) OVER (PARTITION BY company, is_premium ORDER BY month)) / LAG(total_revenue) OVER (PARTITION BY company, is_premium ORDER BY month) * 100, 2) AS revenue_percentage_change,
LEAD(unique_customers) OVER (PARTITION BY company, is_premium ORDER BY month) AS next_month_unique_customers
FROM ganancias_mensuales
WHERE month >= '2023-01' AND month <= '2023-03'
ORDER BY company, is_premium
LIMIT 15;

 * sqlite:///my_data_seccion2.db
Done.


month,company,is_premium,total_revenue,unique_customers,previous_month_revenue,revenue_change,revenue_percentage_change,next_month_unique_customers
2023-01,Company A,0,27306.61,105,None,None,None,96
2023-02,Company A,0,26109.76,96,27306.61,-1196.85,-4.38,112
2023-03,Company A,0,31364.36,112,26109.76,5254.6,20.13,None
2023-01,Company A,1,12624.43,43,None,None,None,46
2023-02,Company A,1,12847.69,46,12624.43,223.26,1.77,49
2023-03,Company A,1,11796.88,49,12847.69,-1050.81,-8.18,None
2023-01,Company B,0,31232.4,111,None,None,None,91
2023-02,Company B,0,24703.14,91,31232.4,-6529.26,-20.91,114
2023-03,Company B,0,32106.77,114,24703.14,7403.63,29.97,None
2023-01,Company B,1,14030.54,59,None,None,None,37


In [77]:
%%sql
WITH active_customers_per_company AS (
    SELECT 
        STRFTIME('%Y-%m', purchase_date) AS month,
        company,
        COUNT(DISTINCT customer_id) AS active_customers,
        CASE WHEN is_premium THEN 'Premium' ELSE 'Non-Premium' END AS customer_type,
        SUM(purchase_amount) AS total_revenue,
        AVG(purchase_amount) AS average_revenue_per_customer
    FROM purchases
    GROUP BY month, company, customer_type
),

monthly_changes AS (
    SELECT
        month,
        company,
        customer_type,
        active_customers,
        total_revenue,
        average_revenue_per_customer,
        LAG(active_customers) OVER (PARTITION BY company, customer_type ORDER BY month) AS previous_month_active_customers,
        active_customers - LAG(active_customers) OVER (PARTITION BY company, customer_type ORDER BY month) AS active_customers_change,
        LAG(total_revenue) OVER (PARTITION BY company, customer_type ORDER BY month) AS previous_month_total_revenue,
        LAG(average_revenue_per_customer) OVER (PARTITION BY company, customer_type ORDER BY month) AS previous_month_average_revenue_per_customer
    FROM active_customers_per_company
),

ranked_companies AS (
    SELECT
        month,
        company,
        customer_type,
        active_customers,
        total_revenue,
        average_revenue_per_customer,
        active_customers_change,
        previous_month_total_revenue,
        previous_month_average_revenue_per_customer,
        RANK() OVER (PARTITION BY month, customer_type ORDER BY active_customers DESC) AS active_customers_rank,
        DENSE_RANK() OVER (PARTITION BY month, customer_type ORDER BY total_revenue DESC) AS total_revenue_rank
    FROM monthly_changes
)

SELECT *
FROM ranked_companies
WHERE month >= '2023-01' AND month <= '2023-03'
ORDER BY month, customer_type, active_customers_rank
LIMIT 20;

 * sqlite:///my_data_seccion2.db
Done.


month,company,customer_type,active_customers,total_revenue,average_revenue_per_customer,active_customers_change,previous_month_total_revenue,previous_month_average_revenue_per_customer,active_customers_rank,total_revenue_rank
2023-01,Company E,Non-Premium,121,30454.57,236.0819379844961,None,None,None,1,2
2023-01,Company B,Non-Premium,111,31232.4,256.0032786885246,None,None,None,2,1
2023-01,Company A,Non-Premium,105,27306.61,246.0054954954955,None,None,None,3,4
2023-01,Company C,Non-Premium,104,29936.22,277.18722222222226,None,None,None,4,3
2023-01,Company D,Non-Premium,99,26445.96,259.2741176470588,None,None,None,5,5
2023-01,Company B,Premium,59,14030.54,233.84233333333336,None,None,None,1,1
2023-01,Company C,Premium,45,11863.73,257.9071739130435,None,None,None,2,3
2023-01,Company E,Premium,44,11335.2,251.89333333333335,None,None,None,3,4
2023-01,Company A,Premium,43,12624.43,280.5428888888889,None,None,None,4,2
2023-01,Company D,Premium,40,10051.79,245.16560975609758,None,None,None,5,5


In [106]:
np.random.seed(42)
df_clientes = pd.DataFrame({
    'customer_id': range(1, 1001),
    'nombre': [f'Cliente_{i}' for i in range(1, 1001)],
    'fecha_registro': [
        datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 300))
        for _ in range(1000)
    ],
    'empresa': np.random.choice(
        ['Company A','Company B','Company C','Company D','Company E'], 1000)
})

print(df_clientes.head())

   customer_id     nombre fecha_registro    empresa
0            1  Cliente_1     2024-04-12  Company A
1            2  Cliente_2     2024-09-27  Company E
2            3  Cliente_3     2024-04-16  Company E
3            4  Cliente_4     2024-03-12  Company C
4            5  Cliente_5     2024-07-07  Company B


In [107]:
df_clientes.to_sql("clientes", con=engine, if_exists="replace", index=False)

1000

In [108]:
%%sql
SELECT 
    c.customer_id,
    c.nombre,
    c.fecha_registro,
    c.empresa,
    COUNT(p.purchase_amount) AS total_purchases,
    SUM(p.purchase_amount) AS total_spent
FROM clientes c
INNER JOIN purchases p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.nombre, c.fecha_registro, c.empresa
ORDER BY total_spent DESC
LIMIT 10;

 * sqlite:///my_data_seccion2.db
Done.


customer_id,nombre,fecha_registro,empresa,total_purchases,total_spent
726,Cliente_726,2024-07-25 00:00:00.000000,Company C,22,6326.53
364,Cliente_364,2024-08-26 00:00:00.000000,Company B,16,5689.79
929,Cliente_929,2024-07-18 00:00:00.000000,Company A,21,5571.1
720,Cliente_720,2024-10-18 00:00:00.000000,Company E,16,5462.33
613,Cliente_613,2024-01-06 00:00:00.000000,Company C,16,5239.28
113,Cliente_113,2024-08-18 00:00:00.000000,Company C,18,5161.9
670,Cliente_670,2024-03-09 00:00:00.000000,Company A,16,5060.7699999999995
104,Cliente_104,2024-01-05 00:00:00.000000,Company E,14,5037.04
822,Cliente_822,2024-05-21 00:00:00.000000,Company E,18,5023.54
283,Cliente_283,2024-04-04 00:00:00.000000,Company B,18,5017.71


In [113]:
%%sql
SELECT 
    c.customer_id,
    c.nombre,
    c.fecha_registro,
    c.empresa,
    COUNT(p.purchase_amount) AS total_purchases,
    SUM(p.purchase_amount) AS total_spent,
    CASE WHEN p.customer_id IS NULL THEN 'No Purchases' ELSE 'Has Purchases' END AS purchase_status
FROM clientes c
LEFT JOIN purchases p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.nombre, c.fecha_registro, c.empresa, purchase_status
ORDER BY total_purchases DESC
LIMIT 10;

 * sqlite:///my_data_seccion2.db
Done.


customer_id,nombre,fecha_registro,empresa,total_purchases,total_spent,purchase_status
726,Cliente_726,2024-07-25 00:00:00.000000,Company C,22,6326.53,Has Purchases
929,Cliente_929,2024-07-18 00:00:00.000000,Company A,21,5571.1,Has Purchases
39,Cliente_39,2024-01-14 00:00:00.000000,Company C,19,3714.55,Has Purchases
73,Cliente_73,2024-06-05 00:00:00.000000,Company D,19,4890.29,Has Purchases
154,Cliente_154,2024-06-20 00:00:00.000000,Company C,19,3500.86,Has Purchases
273,Cliente_273,2024-06-20 00:00:00.000000,Company E,19,4619.29,Has Purchases
776,Cliente_776,2024-01-06 00:00:00.000000,Company B,19,4671.81,Has Purchases
825,Cliente_825,2024-05-18 00:00:00.000000,Company D,19,4019.18,Has Purchases
891,Cliente_891,2024-08-18 00:00:00.000000,Company A,19,4600.2699999999995,Has Purchases
12,Cliente_12,2024-05-31 00:00:00.000000,Company B,18,4307.31,Has Purchases


In [114]:
df_google = df_purchases[df_purchases['company'].isin(['Company A','Company B'])].copy()
df_google['canal'] = 'Google Ads'

df_meta = df_purchases[df_purchases['company'].isin(['Company C','Company D'])].copy()
df_meta['canal'] = 'Meta Ads'

In [115]:
df_google.to_sql("google_ads", con=engine, if_exists="replace", index=False)
df_meta.to_sql("meta_ads", con=engine, if_exists="replace", index=False)

3911

In [116]:
%%sql
WITH all_ads AS (
    SELECT customer_id, company, purchase_date, purchase_amount, canal
    FROM google_ads
    UNION ALL
    SELECT customer_id, company, purchase_date, purchase_amount, canal
    FROM meta_ads
)

SELECT canal, COUNT(*) AS total_purchases, SUM(purchase_amount) AS total_revenue, COUNT(DISTINCT customer_id) AS unique_customers, AVG(purchase_amount) AS average_purchase_amount
FROM all_ads
GROUP BY canal
ORDER BY total_revenue DESC;


 * sqlite:///my_data_seccion2.db
Done.


canal,total_purchases,total_revenue,unique_customers,average_purchase_amount
Google Ads,4052,1049626.64,984,259.03915103652514
Meta Ads,3911,1000029.3,976,255.6965737663002


## Window functions  
   
Agregación — SUM y AVG  
Permiten calcular agregaciones sobre un conjunto de filas relacionadas sin colapsar las filas como lo hace GROUP BY. Cada fila mantiene su valor individual y recibe la agregación como columna adicional.
  
ROW_NUMBER  
Asigna un número único y secuencial a cada fila dentro de una partición. Nunca repite números. Se puede ordenar dentro de la partición de forma ascendente o descendente.  
ROW_NUMBER() OVER (PARTITION BY company ORDER BY purchase_date ASC)  
  
RANK  
Rankea filas dentro de una partición. Si dos filas tienen el mismo valor en la columna de ordenamiento, reciben el mismo número de rank — pero el siguiente número salta según la posición absoluta.  
Ejemplo: 1, 2, 2, 4 — el 3 se salta porque dos filas ocuparon la posición 2  
  
DENSE_RANK  
Funciona igual que RANK pero no salta números aunque haya empates. Los números siempre son consecutivos.  
Ejemplo: 1, 2, 2, 3 — el 3 aparece aunque dos filas ocuparon la posición 2  
  
LAG   
Trae el valor de la fila anterior dentro de la partición. Útil para comparar con el período pasado.  
  
LEAD  
Trae el valor de la fila siguiente dentro de la partición  
  
WITH — Common Table Expression  
Define una tabla temporal nombrada que se usa inmediatamente en la query siguiente. No es permanente — desaparece al terminar la query.  
WITH ventas_mensuales AS (  
    SELECT mes, SUM(amount) as total  
    FROM purchases  
    GROUP BY mes  
)  
SELECT * FROM ventas_mensuales WHERE total > 1000  
  
Tablas Temporales  
Permiten guardar resultados intermedios que persisten durante la sesión — a diferencia de los CTEs que desaparecen al terminar la query. Útiles cuando necesitas reutilizar el mismo resultado en múltiples queries distintas.  
CREATE TEMPORARY TABLE clientes_segmentados AS  
SELECT ...  
  
INNER JOIN  
Interseccion entre ambas tablas  
  
LEFT JOIN (FROM)  
Todos de la tabla izquierda, null en columnas de la derecha si no tienen coincidencia  
  
RIGHT JOIN  
Todos de la tabla derecha, null en columnas de la izquierda si no tienen coincidencia  
  
UNION  
Une filas de dos tablas verticalmente — a diferencia del JOIN que relaciona columnas horizontalmente.  
UNION - Distintos
UNION ALL - Todos con duplicados